In [36]:
import json

from dotenv import load_dotenv
load_dotenv()

import json
import re
from anthropic.types import TextBlock, ThinkingBlock

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"


In [34]:
def remove_json_markdown(response: str) -> str:
    ''' Removes the Markdown formatting from a JSON string '''

    m = re.search(r'^```json\n(.+)```$', response, re.DOTALL) #, re.MULTILINE | re.DOTALL)
    m1 = m.group(1)
    return m1

def response_content_text(response_content: list) -> str:

    if isinstance(response_content[0], TextBlock):
        print("The response contains a TextBlock.")
        response_text = response_content[0].text

    elif isinstance(response_content[0], ThinkingBlock):
        print("The response contains a ThinkingBlock.")
        removed_json_markdown = remove_json_markdown(response_content[1].text)
        response_text = removed_json_markdown

    print(f'RESPONSE TEXT: {response_text}')
    return response_text


In [33]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None): # stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages
    }
    if system:
        params["system"] = system
    
    response = client.messages.create(**params)
    return response_content_text(response.content)

In [ ]:
def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
[
  {
    "task": "Description of task",
  },
  ...additional
]

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []

  add_user_message(messages, prompt)
  text = chat(messages) #, stop_sequences=["```"])
  #text = response.content[0].text 
  #print(text)
  return json.loads(text)



In [ ]:
dataset = generate_dataset()
print(dataset)

In [38]:
with open('dataset_11.json', 'w') as f:
    json.dump(dataset, f, indent=2)